# Transformer Foundations, Part 1 of 3: Attention and Transformer Blocks

> **Series route:** Part 1 builds attention and the reusable Transformer block. [Part 2](02-decoder-only-language-model.ipynb) turns that block into a causal language model. [Part 3](03-encoder-decoder-and-cross-attention.ipynb) adds bidirectional encoding and cross-attention.

The original `the cat sat on the mat` running example remains unchanged. This notebook stops at the full Pre-LN Transformer block so the mechanism can settle before training a language model.


# Transformer Foundations: Three-Part Series Map

## Building Intuition One Step at a Time

This three-part series builds the **complete mental model for the Transformer architecture** from first principles - starting with a tiny 3-dimensional semantic space that you can visualise, rotate, and reason about concretely.

Every concept across the three notebooks is demonstrated on the same running example:

> **"the cat sat on the mat"**

| Step | Concept | Key Idea |
| ---- | ------- | -------- |
| 1  | Vocabulary + 3D Embeddings     | Words as points in semantic space |
| 2  | The Ordering Problem           | Why bags of words lose meaning |
| 3  | Sinusoidal PE                  | Adding position with sine waves |
| 4  | RoPE                           | Rotating Q/K vectors for relative position |
| 5  | Q, K, V + Attention            | Soft dictionary lookup |
| 6  | Multi-Head Attention           | Parallel attention heads |
| 7  | Feed-Forward + Layer Norm      | Per-token transformation + stabilisation |
| 8  | Full Transformer Block         | All components assembled |
| 9  | Mini Language Model            | End-to-end training from scratch |
| 10 | W_V as Relevance Filter        | What each token contributes to the blend |
| 11 | Causal Triangle                | Layer stacking and last-position richness |
| 12 | Encoder Architecture           | Bidirectional attention - mask=None |
| 13 | Cross-Attention                | Q from decoder, K/V from encoder |
| 14 | Encoder-Decoder                | Reversal task with cross-attention |
| 15 | Architecture Comparison        | Reader, Writer, Translator side by side |
| 16 | GPT-2 Internals                | A real model, cracked open |

**Scope note:** this series builds real, measured intuition for the full mechanism space above; a few adjacent engineering topics (KV-caching, other positional-encoding schemes, decoding variants) are named but not built - see the series-wide tier breakdown at the end of Part 3.


![The Transformer Foundations journey across this three-part series, from raw tokens to a live pretrained Transformer](images/transformer-learning-journey.png)

---

## Prerequisite Bridge — From PyTorch Foundations and `01-rnns`

Run these in order before this notebook:

1. `00-pytorch-fundamentals/01-keras-to-pytorch-antarctic-field-guide.ipynb` — translates Keras training habits into PyTorch: `nn.Module`, raw logits, `.backward()`, the explicit optimizer loop, and tensor/device contracts.
2. `01-rnns/01-pytorch-rnn-bridge.ipynb` — applies those mechanics to sequence tensors: integer token IDs, `(B,T) → (B,T,D)`, recurrent outputs/states, `(B,T,V)` logits, padding-aware loss, and autoregressive generation.

| Foundation | Role in this notebook |
|---|---|
| `nn.Module` subclassing, explicit training loop | Every mechanism here (`MultiHeadAttention`, `FeedForward`, `TransformerBlock`) is an `nn.Module`; training uses the identical 4-step loop |
| Computation graph and `.backward()` | Gradients flow through Q·Kᵀ/√dₖ→softmax→V; the same chain-rule traced on `y=x²` now runs through attention layers |
| Autograd warm-up (`y = x²`, parabola minimisation) | The gradient-descent intuition built on the toy parabola carries over directly to the loss surface here |
| Tensor shapes: `(batch, features)` → `(batch, seq, dim)` | The shape contract expands by one axis; every `(B, S, D)` shape annotation in this notebook assumes comfort with the RNN bridge's sequence exercises |
| Token IDs, embeddings, raw sequence logits, and next-token loss | The MiniLM preserves the same `(B,T) → (B,T,D) → (B,T,V)` contract while replacing recurrent state with attention |

> **If you have not run both notebooks** the autograd and sequence-shape patterns used here will feel unfamiliar. Complete the foundations lab first, then the PyTorch RNN bridge.

In [ ]:
#  Install dependencies (run once)
import subprocess, sys

required = [
    ("numpy",        "numpy"),
    ("matplotlib",   "matplotlib"),
    ("torch",        "torch"),
    ("seaborn",      "seaborn"),
    ("plotly",       "plotly"),
    ("transformers", "transformers"),
]

# Import each required package, installing it via pip only if it's missing
for imp, pkg in required:
    try:
        __import__(imp)
        print(f"  ok  {pkg}")
    except ImportError:
        print(f"  installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"  done {pkg}")


> **PyTorch → Keras:** `import torch`, `import torch.nn as nn`, `import torch.nn.functional as F` — the three imports this notebook builds on: `torch` for tensors/autograd, `nn` for layer/module building blocks, `F` for stateless functional ops (softmax, GELU, etc.). **Keras/TF equivalent:** `import tensorflow as tf`; layers come from `tf.keras.layers`, functional ops from `tf.nn` (e.g. `tf.nn.softmax`) — TF/Keras doesn't split "module" vs "functional" APIs into two separate top-level namespaces the way PyTorch's `nn` vs `nn.functional` does.

In [ ]:
#  Imports
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import math, warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import seaborn as sns
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation

warnings.filterwarnings('ignore')

# Plotly is optional -- fall back to matplotlib-only 3D plots if it isn't installed
try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

plt.rcParams.update({"figure.dpi": 100, "figure.facecolor": "white"})
print(f"torch   {torch.__version__}")
print(f"numpy   {np.__version__}")


---

## Part 1 - Our Mini Universe: Vocabulary & Embeddings

Before anything else, we need a way to represent words as numbers. This is the job of an **embedding**.

In a real model (e.g. GPT-2), each word lives in a 768-dimensional space. Instead we build a **3-dimensional** semantic space where each axis captures a meaningful property:

| Axis | Meaning | Low (0) | High (1) |
| ---- | ------- | ------- | -------- |
| **dim 0** | Concreteness | abstract (articles) | physical objects (cat, mat) |
| **dim 1** | Animacy | inanimate (mat, fence) | living beings (cat, dog) |
| **dim 2** | Dynamism | static (mat, fence) | action words (ran, jumped) |


> **PyTorch → Keras:** `torch.tensor([...], dtype=torch.float32)` builds the 3D embedding matrix from a Python list. **Keras/TF equivalent:** `tf.constant([...], dtype=tf.float32)` (or `tf.convert_to_tensor`) — both frameworks build an immutable tensor from raw values the same way; a real model would instead use a learnable `nn.Embedding`/`tf.keras.layers.Embedding` table.

In [ ]:
#  Vocabulary
VOCAB = {
    "<PAD>": 0, "<BOS>": 1, "<EOS>": 2,
    "the": 3, "a": 4, "cat": 5, "dog": 6,
    "mat": 7, "fence": 8, "sat": 9, "ran": 10,
    "jumped": 11, "on": 12, "over": 13, "big": 14,
}

# Invert the vocab so token IDs can be decoded back to words
IDX2WORD = {v: k for k, v in VOCAB.items()}
VOCAB_SIZE = len(VOCAB)

#  3D Semantic Embeddings (Concreteness, Animacy, Dynamism)
E = {
    "<PAD>": [0.00, 0.00, 0.00], "<BOS>": [0.08, 0.08, 0.15], "<EOS>": [0.08, 0.08, 0.15],
    "the":   [0.05, 0.04, 0.08], "a":     [0.05, 0.04, 0.08],
    "cat":   [0.91, 0.94, 0.38], "dog":   [0.88, 0.92, 0.55],
    "mat":   [0.96, 0.04, 0.04], "fence": [0.93, 0.03, 0.03],
    "sat":   [0.34, 0.18, 0.78], "ran":   [0.28, 0.12, 0.96],
    "jumped":[0.30, 0.14, 0.98], "on":    [0.14, 0.04, 0.18],
    "over":  [0.17, 0.04, 0.24], "big":   [0.44, 0.04, 0.09],
}

# Stack each token's embedding vector in vocab-ID order into one matrix
embedding_matrix = torch.tensor(
    [E[IDX2WORD[i]] for i in range(VOCAB_SIZE)], dtype=torch.float32
)

SENTENCE = "the cat sat on the mat"
TOKENS = SENTENCE.split()

# Encode the running example sentence into vocab IDs
TOKEN_IDS = [VOCAB[w] for w in TOKENS]
SEQ_LEN = len(TOKENS)

print(f"Vocab size       : {VOCAB_SIZE}")
print(f"Embedding shape  : {tuple(embedding_matrix.shape)}  (vocab x 3D)")
print(f"Running sentence : {SENTENCE!r}")
print(f"Token IDs        : {TOKEN_IDS}")
print()
print("Embedding matrix -> Concreteness, Animacy, Dynamism:")

# Skip the special tokens (indices 0-2) when printing the embedding table
for word, vec in list(E.items())[3:]:
    print(f"  {word:<10} [{vec[0]:.2f}, {vec[1]:.2f}, {vec[2]:.2f}]")


In [ ]:
#  Interactive 3D Vocabulary Visualisation
# Plotly = interactive (drag to rotate). Matplotlib = static fallback.

CATEGORIES = {
    "Article":        (["the", "a"],            "#636EFA"),
    "Animate Noun":   (["cat", "dog"],           "#00CC96"),
    "Inanimate Noun": (["mat", "fence"],         "#AB63FA"),
    "Verb":           (["sat", "ran", "jumped"], "#EF553B"),
    "Preposition":    (["on", "over"],           "#FFA15A"),
    "Adjective":      (["big"],                  "#19D3F3"),
}

# Prefer the interactive Plotly scatter when it's available
if HAS_PLOTLY:
    fig = go.Figure()

    # Draw each word category as its own colored cluster of points
    for cat, (words, color) in CATEGORIES.items():
        xs, ys, zs = zip(*[E[w] for w in words])
        fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode='markers+text', text=words,
            textposition='top center', name=cat,
            marker=dict(size=10, color=color, opacity=0.85, line=dict(color='white', width=1))))

    # Trace the running sentence as a connected path through embedding space
    sx, sy, sz = zip(*[E[w] for w in TOKENS])
    fig.add_trace(go.Scatter3d(x=sx, y=sy, z=sz, mode='lines', name='Sentence path',
        line=dict(color='gold', width=4, dash='dot')))
    fig.update_layout(
        title=dict(text='<b>3D Semantic Embedding Space</b> - drag to rotate', x=0.5),
        scene=dict(xaxis_title='Concreteness', yaxis_title='Animacy', zaxis_title='Dynamism'),
        width=820, height=560)
    fig.show()
else:

    # Fall back to a static matplotlib 3D scatter when Plotly isn't installed
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection='3d')
    cmap = {'Article': 'royalblue', 'Animate Noun': 'mediumseagreen',
            'Inanimate Noun': 'mediumpurple', 'Verb': 'tomato',
            'Preposition': 'darkorange', 'Adjective': 'deepskyblue'}

    # Draw each category's points, then label every word next to its marker
    for cat, (words, _) in CATEGORIES.items():
        xs, ys, zs = zip(*[E[w] for w in words])
        ax.scatter(xs, ys, zs, s=90, label=cat, color=cmap[cat], alpha=0.9)
        for w in words:
            ax.text(E[w][0], E[w][1], E[w][2], f' {w}', fontsize=9)

    # Trace the running sentence as a connected path through embedding space
    sx, sy, sz = zip(*[E[w] for w in TOKENS])
    ax.plot(sx, sy, sz, 'o--', color='gold', lw=2, label='Sentence path')
    ax.set_xlabel('Concreteness'); ax.set_ylabel('Animacy'); ax.set_zlabel('Dynamism')
    ax.set_title('3D Semantic Embedding Space'); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()
    print('Tip: pip install plotly for an interactive, rotatable version')


### Tokenisation

A **tokeniser** converts a raw string into integer IDs the model can process. In our toy system, one word = one token. Production models use sub-word tokenisation (BPE / SentencePiece).


In [ ]:
#  Tokeniser
# Convert a raw string into vocab IDs, optionally wrapping with BOS/EOS markers
def encode(text: str, add_bos: bool = False, add_eos: bool = False):
    ids = [VOCAB.get(w, VOCAB["<PAD>"]) for w in text.lower().split()]
    if add_bos:
        ids = [VOCAB["<BOS>"]] + ids
    if add_eos:
        ids = ids + [VOCAB["<EOS>"]]
    return ids


# Convert vocab IDs back into a whitespace-joined string
def decode(ids):
    return " ".join(IDX2WORD.get(i, "<?>") for i in ids)


phrase = "the big cat jumped over the fence"
enc = encode(phrase)
dec = decode(enc)
print(f"Input  : {phrase!r}")
print(f"Encoded: {enc}")
print(f"Decoded: {dec!r}")
print()
print('With BOS/EOS markers:')
enc2 = encode(phrase, add_bos=True, add_eos=True)
print(f"  {enc2}")
print()
print("  -> One word = one token; BPE splits rare words in real models.")


---

## Attention: First Contact

Before positions, before $Q/K/V$ projections, before multi-head - the beating heart of the transformer is one simple idea:

> **Every token looks at every other token and builds a weighted average of them, where the weights answer "how much do I care about you?"**

**Running it on our sentence:** `"the cat sat on the mat"`. Query = `"cat"`. No positions, no learned projections - just raw dot products of the 3D semantic vectors.


In [ ]:
#  Attention, step by step - freezing at each completed step
# Minimal attention: query = key = value = the raw embedding.

emb_min = embedding_matrix[TOKEN_IDS].numpy()   # (S, 3)
S_min = len(TOKENS)
QUERY = "cat"
qi = TOKENS.index(QUERY)

# Score every token against the query by raw dot product
scores_min = emb_min @ emb_min[qi]

# Manual softmax: shift for numerical stability, then normalise to sum to 1
w_min = np.exp(scores_min - scores_min.max())
w_min /= w_min.sum()

# Weighted sum of embeddings = the attention output
output_min = (w_min[:, None] * emb_min).sum(0)

key_x = np.arange(S_min)
q_x = (S_min - 1) / 2.0
dim_names = ["Concrete", "Animate", "Dynamic"]

PH, REVEAL, HOLD = 4, 18, 12
plen = REVEAL + HOLD
TOTAL = PH * plen + 18


# Map an animation frame index to (phase, reveal-progress, is-phase-complete)
def _phase(f):
    if f >= PH * plen:
        return PH - 1, 1.0, True
    p = f // plen
    loc = f % plen
    return p, min(loc / REVEAL, 1.0), loc >= REVEAL


# Layout: top row spans both columns (token graph), bottom row splits into weight/output panels
fig = plt.figure(figsize=(12, 7))
gs = plt.GridSpec(2, 2, height_ratios=[1.5, 1], hspace=0.5, wspace=0.25)
ax_g = fig.add_subplot(gs[0, :])
ax_w = fig.add_subplot(gs[1, 0])
ax_o = fig.add_subplot(gs[1, 1])

captions = [
    'Step 1 - pick a query token: "cat" asks "who matters to me?"',
    'Step 2 - score "cat" against every token by dot product',
    'Step 3 - softmax turns scores into weights that sum to 1',
    'Step 4 - output = weighted sum of the value vectors',
]
done_caps = [
    'Step 1 - query selected', 'Step 2 - every token scored',
    'Step 3 - weights sum to 1', 'Step 4 - context-aware vector for "cat" is ready',
]


# Redraw all three panels (token graph, weight bars, output bars) for animation frame f
def update(f):
    p, t, done = _phase(f)
    ax_g.clear(); ax_w.clear(); ax_o.clear()
    ax_g.set_xlim(-1, S_min); ax_g.set_ylim(-0.6, 1.7); ax_g.axis('off')
    if p >= 1:

        # Draw attention lines from query to each key, thickness/opacity scaled by score
        wshow = (scores_min - scores_min.min()) / max((scores_min.max() - scores_min.min()), 1e-9)
        reveal = t if p == 1 else 1.0
        for j in range(S_min):
            lw = 0.5 + 6 * wshow[j] * reveal
            a = min(0.15 + 0.85 * wshow[j] * reveal, 1.0)
            ax_g.plot([q_x, key_x[j]], [0, 1], color='#4c72b0', lw=lw, alpha=a, zorder=1)

    # Draw every token as a node, annotating its attention weight once scores exist
    for j, tok in enumerate(TOKENS):
        ax_g.scatter(key_x[j], 1, s=520, color='#dddddd', edgecolor='#888', zorder=3)
        ax_g.text(key_x[j], 1, tok, ha='center', va='center', fontsize=9, zorder=4)
        if p >= 2:
            ax_g.text(key_x[j], 1.32, f'{w_min[j]:.2f}', ha='center', fontsize=9, color='#c44e52', fontweight='bold')
    qsize = 300 + 420 * (t if p == 0 else 1.0)
    ax_g.scatter(q_x, 0, s=qsize, color='gold', edgecolor='#b8860b', zorder=5)
    ax_g.text(q_x, 0, QUERY, ha='center', va='center', fontsize=10, fontweight='bold', zorder=6)
    ax_g.text(q_x, -0.42, 'query', ha='center', fontsize=9, color='#b8860b')
    ax_g.text((S_min-1)/2, 1.6, 'keys / values (every token)', ha='center', fontsize=9, color='#555')
    ax_w.set_xlim(-0.6, S_min-0.4); ax_w.set_ylim(0, 1.05)
    ax_w.set_xticks(key_x); ax_w.set_xticklabels(TOKENS, fontsize=8, rotation=20)
    if p == 0:
        ax_w.set_title('scores appear in step 2', fontsize=9, color='#999')
    elif p == 1:
        sc = (scores_min - scores_min.min()) / max(scores_min.max()-scores_min.min(), 1e-9)
        ax_w.bar(key_x, sc * t, color='#4c72b0', alpha=0.85)
        ax_w.set_title('Step 2 - raw dot-product scores', fontsize=10)
        ax_w.set_ylabel('score (scaled)')
    else:
        sc = (scores_min-scores_min.min()) / max(scores_min.max()-scores_min.min(), 1e-9)
        blend = (1-t)*sc + t*w_min if p == 2 else w_min
        colors = ['gold' if j==w_min.argmax() else '#4c72b0' for j in range(S_min)]
        ax_w.bar(key_x, blend, color=colors, alpha=0.85)
        ax_w.set_title('Step 3 - softmax -> weights (sum=1)' if p==2 else 'Step 3 - attention weights', fontsize=10)
        ax_w.set_ylabel('weight')
    ax_o.set_ylim(0, 1.05); ax_o.set_xticks(range(3)); ax_o.set_xticklabels(dim_names, fontsize=8)
    if p < 3:
        ax_o.bar(range(3), [0,0,0], color='#55a868'); ax_o.set_title('output builds in step 4', fontsize=9, color='#999')
    else:

        # Build the output vector incrementally, revealing one weighted token contribution at a time
        frac=t*S_min; k=int(frac); part=frac-k
        out_v=np.zeros(3)
        for j in range(min(k, S_min)): out_v += w_min[j]*emb_min[j]
        if k < S_min: out_v += part*w_min[k]*emb_min[k]
        ax_o.bar(range(3), out_v, color='#55a868')
        cur=TOKENS[min(k, S_min-1)]
        ax_o.set_title(f'Step 4 - sum w*value (adding "{cur}")' if not done else 'Step 4 - context vector', fontsize=10)
    fig.suptitle(done_caps[p] if done else captions[p], fontsize=12, fontweight='bold', color='#333')


# Assemble the frame-by-frame callback into a playable, looping animation
ani = FuncAnimation(fig, update, frames=TOTAL, interval=45, blit=False, repeat=True, repeat_delay=1200)
plt.close(fig)
print('Minimal attention: query = key = value = embedding, no positions, no projections.')
top3 = w_min.argsort()[::-1][:3]
print('"cat" attends most to: ' + ", ".join(f"{TOKENS[j]} ({w_min[j]:.0%})" for j in top3))


#### What just happened - and what's missing

`"cat"` pulled most strongly toward **itself** and **`"dog"`** - its semantic neighbours. Attention found *meaning* without anyone hand-coding grammar.

But look closely at what we **never used**: *position*. Query, key and value were the raw embeddings. Shuffle the sentence and `"cat"` keeps the exact same neighbours. **Attention, on its own, is position-blind.**


---

## Part 2 - The Ordering Problem

What happens if we just **sum or average** the token vectors?

> "the cat sat on the mat"
> "mat the on sat the cat" - shuffled nonsense

Both sentences contain exactly the same words. Their mean-pooled embedding is **identical** - the model cannot tell them apart. Position information is load-bearing.


> **PyTorch → Keras:** `embedding_matrix[ids].mean(dim=0)` indexes the embedding tensor by token ids then mean-pools across the sequence axis (`dim=0`); `torch.allclose(...)` checks two tensors are numerically equal within tolerance. **Keras/TF equivalent:** `tf.reduce_mean(tf.gather(embedding_matrix, ids), axis=0)` and `tf.experimental.numpy.allclose` / `np.allclose` — PyTorch's `dim=` argument is the same concept as TF's `axis=`.

In [ ]:
#  Bag of Words - loses all positional information
def bag_of_words(sentence: str) -> torch.Tensor:
    """Mean-pool embeddings - loses all positional information."""
    ids = torch.tensor(encode(sentence), dtype=torch.long)
    return embedding_matrix[ids].mean(dim=0)


sentences = [
    "the cat sat on the mat",
    "mat the on sat the cat",
    "sat cat mat on the the",
]
print('Mean-pooled vectors (all contain the same words):')
for s in sentences:
    v = bag_of_words(s).numpy()
    print(f"  {s!r:<42}  [{v[0]:.3f}, {v[1]:.3f}, {v[2]:.3f}]")

# Check whether every word-order permutation collapses to the same mean-pooled vector
all_same = all(
    torch.allclose(bag_of_words(sentences[0]), bag_of_words(s))
    for s in sentences[1:]
)
print(f"\nAll three vectors identical: {all_same}")
print()
print('  -> A model with no positional encoding treats meaningful sentences and')
print('     complete nonsense as the SAME input.  We need positional encoding.')


---

## Part 3 - Positional Encoding

### 3a. Sinusoidal PE (original Transformer, "Attention Is All You Need")

$$PE_{(m,\, 2i)} = \sin\!\left(\frac{m}{10000^{2i/d}}\right)$$

$$PE_{(m,\, 2i+1)} = \cos\!\left(\frac{m}{10000^{2i/d}}\right)$$

Each dimension pair oscillates at a different frequency - a unique fingerprint for every position.


> **PyTorch → Keras:** `torch.tensor(pe)` wraps the numpy-computed sinusoidal table into a tensor, and `emb_vectors + pe_3d` is elementwise tensor addition. **Keras/TF equivalent:** `tf.constant(pe)` and plain `+` on TF tensors — elementwise arithmetic operators are overloaded identically in both frameworks.

In [ ]:
#  Sinusoidal Positional Encoding
def sinusoidal_pe(seq_len: int, d_model: int) -> torch.Tensor:
    """Classic additive positional encoding (Vaswani et al. 2017).
    Handles both even and odd d_model gracefully.
    """
    pe = np.zeros((seq_len, d_model), dtype=np.float32)
    positions = np.arange(seq_len)[:, None].astype(np.float32)
    dims = np.arange(0, d_model, 2).astype(np.float32)

    # Geometrically decaying frequency per dimension pair
    freqs = 1.0 / (10000 ** (dims / d_model))

    # Even dims get sine, odd dims get cosine, at the same frequency
    pe[:, 0::2] = np.sin(positions * freqs)
    n_cos = pe[:, 1::2].shape[1]
    pe[:, 1::2] = np.cos(positions * freqs[:n_cos])
    return torch.tensor(pe)


D_VIS = 16
pe_matrix = sinusoidal_pe(SEQ_LEN, D_VIS)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Heatmap: PE value at every (token position, dimension) pair
ax = axes[0]
im = ax.imshow(pe_matrix.numpy(), aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
ax.set_xticks(range(D_VIS))
ax.set_xticklabels([f'd{i}' for i in range(D_VIS)], fontsize=8, rotation=45)
ax.set_yticks(range(SEQ_LEN)); ax.set_yticklabels(TOKENS, fontsize=10)
ax.set_title('Sinusoidal PE - our sentence')
ax.set_xlabel('Embedding dimension'); ax.set_ylabel('Token position')
plt.colorbar(im, ax=ax)

# Line plot: a few dimensions over 50 positions, revealing fast vs. slow oscillation
ax2 = axes[1]
pe_long = sinusoidal_pe(50, D_VIS).numpy()
for i in [0, 2, 6, 14]:
    label = f'dim {i} - {"fast" if i < 4 else "slow"}'
    ax2.plot(pe_long[:, i], label=label, lw=1.8)
ax2.set_title('PE signal per dimension over 50 positions')
ax2.set_xlabel('Token position'); ax2.set_ylabel('PE value')
ax2.legend(fontsize=8); ax2.set_ylim(-1.1, 1.1)

plt.suptitle('Sinusoidal Positional Encoding', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Add the 3D positional signal directly onto the token embeddings
emb_vectors = embedding_matrix[TOKEN_IDS]
pe_3d = sinusoidal_pe(SEQ_LEN, 3)
enriched = emb_vectors + pe_3d
print('After adding 3D sinusoidal PE:')
for i, w in enumerate(TOKENS):

    # Format a 3-vector for aligned printing
    def fmt(v_list):
        return f'[{v_list[0]:+.3f}, {v_list[1]:+.3f}, {v_list[2]:+.3f}]'
    print(f'[{i}] {w:<8}  orig={fmt(emb_vectors[i].tolist())}  pe={fmt(pe_3d[i].tolist())}  sum={fmt(enriched[i].tolist())}')


#### A nagging question before we move on

Sinusoidal PE looks great in the heatmap - so why did the field move to RoPE? **Position is injected at the input, but attention projects through $W_Q$ first. If that projection smears the position signal, it was partially wasted.** RoPE fixes this by injecting position *after* the $W_Q$ projection, directly into the dot-product.


> **PyTorch → Keras:** `torch.manual_seed(0)` seeds PyTorch's global RNG; `torch.randn(...)` samples standard-normal tensors; `x_in @ W_Q_demo.T` is matrix multiplication (`@` calls `torch.matmul` under the hood) with an explicit `.T` transpose. **Keras/TF equivalent:** `tf.random.set_seed(0)`, `tf.random.normal(...)`, and `tf.matmul(x_in, W_Q_demo, transpose_b=True)` — TF's `matmul` takes a `transpose_b` flag instead of pre-transposing the second operand.

In [ ]:
#  WHY not just keep sinusoidal PE? Watch the clean signal dilute
torch.manual_seed(0)
d_demo = 16; seq_demo = 12

pe_demo = sinusoidal_pe(seq_demo, d_demo)
content = torch.randn(seq_demo, d_demo)
W_Q_demo = torch.randn(d_demo, d_demo) * (1 / math.sqrt(d_demo))

# Simulate content+PE summed at the input, then projected through a random W_Q
x_in = content + pe_demo
q_proj = x_in @ W_Q_demo.T


# Cosine-similarity matrix between every pair of position vectors
def pos_sim(mat):
    mat = mat.detach().numpy() if isinstance(mat, torch.Tensor) else np.asarray(mat)
    m = mat / (np.linalg.norm(mat, axis=-1, keepdims=True) + 1e-9)
    return m @ m.T


sim_pe = pos_sim(pe_demo)
sim_q  = pos_sim(q_proj)

# Compare the cosine-similarity structure before vs. after the W_Q projection
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax_s, sim, title in [(axes[0], sim_pe, 'PURE sinusoidal PE\nclean diagonal band = distance-aware'),
                          (axes[1], sim_q,  'AFTER (content+PE) @ W_Q  (RANDOM W_Q)\nstructure can wash out')]:
    sns.heatmap(sim, ax=ax_s, cmap='RdBu_r', vmin=-1, vmax=1, square=True, cbar_kws={'label': 'cosine sim'})
    ax_s.set_title(title); ax_s.set_xlabel('position'); ax_s.set_ylabel('position')
plt.suptitle('Sinusoidal PE can dilute once it is summed with content and projected', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


# Correlate similarity with position closeness -- higher = cleaner distance-aware structure
def monotonicity(sim):
    n = sim.shape[0]
    closeness = np.array([[-abs(i-j) for j in range(n)] for i in range(n)])
    return np.corrcoef(closeness.flatten(), sim.flatten())[0, 1]


print('"Closer positions = more similar" correlation:')
print(f'  Pure PE (at the input)         : {monotonicity(sim_pe):+.3f}   <- strong, clean')
print(f'  After content + W_Q projection : {monotonicity(sim_q):+.3f}   <- weaker')
print()
print('The principled case for RoPE: position is injected AFTER projection,')
print('so W_Q cannot dilute it, and the Q.K score depends only on (m-n) BY CONSTRUCTION.')


![Positional encoding strategies: fixed sinusoidal vs. rotary (RoPE) relative encoding](images/positional-encoding-and-rope.png)

### 3b. RoPE - Rotary Positional Embeddings

RoPE **rotates** the Query and Key vectors just before the dot-product, by an angle that depends on absolute position. The rotation cancels in a relative way - only the gap $m - n$ survives.

$$\theta_i = \frac{1}{10000^{2i/d}}$$

Token at position $m$ gets its $i$-th pair rotated by angle $m \cdot \theta_i$:

$$\text{RoPE}(x, m)_{2i:2i+2} = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix} \begin{pmatrix} x_{2i} \\ x_{2i+1} \end{pmatrix}$$


In [ ]:
#  RoPE theta values - frequency decay visualisation
D_ROPE = 2
half = D_ROPE // 2

# theta_i decays geometrically with dimension-pair index i
thetas = np.array([1.0 / (10000 ** (2 * i / D_ROPE)) for i in range(half)])
print(f'theta values for d={D_ROPE}: {thetas}')

steps = 50

# Accumulated rotation angle = position * theta, for every position up to 50
angles = np.outer(np.arange(steps), thetas)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Plot each pair's cosine component across positions
ax = axes[0]
for i in range(half):
    ax.plot(np.cos(angles[:, i]), label=f'Pair {i} (theta={thetas[i]:.4f})', lw=1.8)
ax.set_xlabel('Token position'); ax.set_ylabel('cos(m * theta_i)')
ax.set_title('RoPE: cosine component per dimension pair over 50 positions')
ax.legend(fontsize=8); ax.set_ylim(-1.1, 1.1)

# Plot each pair's raw accumulated angle, wrapped to [0, 2*pi)
ax2 = axes[1]
for i in range(half):
    ax2.plot(range(steps), angles[:, i] % (2 * np.pi), label=f'Pair {i}', lw=1.8)
ax2.set_xlabel('Token position'); ax2.set_ylabel('angle mod 2pi')
ax2.set_title('Accumulated rotation angle\nPair 0 spins fastest, pair 2 slowest')
ax2.legend(fontsize=8)

plt.suptitle('RoPE theta values: high-frequency pairs capture local position', fontsize=10, fontweight='bold')
plt.tight_layout(); plt.show()


### 3d. Building the RoPE animation the way you'd actually discover it

Nobody arrives at a good visualisation in one shot. We build it in front of you, refinement by refinement, each step motivated by a genuine complaint about the one before.

**Step 1 - the crudest possible picture:** just plot one pair (pair 0) as a clock dial.


#### Steps 2, 3 and 4 - all at once, because the complaints compound

- **Step 2 (stacking)** - each of the 3 pairs gets its own disc at its own height.
- **Step 3 (one tower per token)** - a column per position rather than a single position.
- **Step 4 (labels)** - degree labels on each disc.


In [ ]:
#  Attempt 1 - the crudest possible RoPE picture: one dial
th0 = 1.0 / (10000 ** (0 / 6))
circle = np.linspace(0, 2 * np.pi, 100)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4), subplot_kw={'aspect': 'equal'})

# Draw one dial per position, arrow angle = position * theta_0
for ax, m in zip(axes, range(4)):
    ang = m * th0
    ax.plot(np.cos(circle), np.sin(circle), 'lightgray', lw=1)
    ax.arrow(0, 0, math.cos(ang)*0.85, math.sin(ang)*0.85,
             head_width=0.1, head_length=0.08, fc='#4c72b0', ec='#4c72b0', lw=2)
    ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)
    ax.set_title(f'position {m}\nangle = {math.degrees(ang):.1f}°', fontsize=9)
    ax.axis('off')

plt.suptitle(f'Pair 0: angle grows by theta_0 = {th0:.4f} rad per step', fontsize=10, fontweight='bold')
plt.tight_layout(); plt.show()
print('Complaint: we only see one pair out of three, and only one position at a time.')


In [ ]:
#  Animated RoPE - watch each pair rotate as position increases
RADII = [1.0, 0.7, 0.4]
COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c']
HEIGHTS = [1.2, 0.6, 0.0]
N_FRAMES = 60

fig3d = plt.figure(figsize=(7, 9))
ax3d = fig3d.add_subplot(111, projection='3d')


# Per-frame callback: redraw all three dimension pairs' rotation discs at position step_m
def rope_frame(step_m):
    ax3d.clear()
    circle_t = np.linspace(0, 2 * np.pi, 200)

    # Each pair rotates at its own theta_i -- draw its disc and current angle arrow
    for pair_i, (r, col, h) in enumerate(zip(RADII, COLORS, HEIGHTS)):
        theta_i = 1.0 / (10000 ** (2 * pair_i / D_ROPE))
        ang = step_m * theta_i
        ax3d.plot(r*np.cos(circle_t), r*np.sin(circle_t), h, color=col, lw=0.5, alpha=0.3)
        ax3d.quiver(0, 0, h, r*math.cos(ang), r*math.sin(ang), 0, color=col, lw=2, arrow_length_ratio=0.15)
        ax3d.text(r*1.05, 0, h, f'Pair {pair_i}', fontsize=8, color=col)
    ax3d.set_xlim(-1.3, 1.3); ax3d.set_ylim(-1.3, 1.3); ax3d.set_zlim(-0.3, 1.7)
    ax3d.set_xlabel('cos', fontsize=7); ax3d.set_ylabel('sin', fontsize=7)
    ax3d.set_title(f'RoPE rotation at position m={step_m}', fontsize=10)
    ax3d.tick_params(labelsize=7)


# Animate the rotation across positions 0..59
ani3d = FuncAnimation(fig3d, rope_frame, frames=N_FRAMES, interval=80, repeat=True)
plt.close(fig3d)
print('Each disc = one dimension pair. Arrow angle = m * theta_i.')
print('Pair 0 (blue) spins fastest; pair 2 (green) barely moves.')
HTML(ani3d.to_jshtml(default_mode='loop'))


In [ ]:
#  RoPE implementation + relative-distance proof

def rope_rotate(x, m: int, thetas_arr: np.ndarray) -> np.ndarray:
    """Rotate vector x (shape: d) at token position m using RoPE."""
    x_rot = np.array(x, dtype=np.float32).copy()

    # Rotate each dimension pair by its own accumulated angle (m * theta_i)
    for i, th in enumerate(thetas_arr):
        angle = m * th
        c, s = math.cos(angle), math.sin(angle)
        a, b = x_rot[2*i], x_rot[2*i+1]
        x_rot[2*i]   = a*c - b*s
        x_rot[2*i+1] = a*s + b*c
    return x_rot


thetas_demo = np.array([1.0 / (10000 ** (2*i/D_ROPE)) for i in range(D_ROPE//2)])
cat_emb = embedding_matrix[VOCAB['cat']].numpy()[:D_ROPE]
mat_emb = embedding_matrix[VOCAB['mat']].numpy()[:D_ROPE]

print('Relative-distance property of RoPE:')
print('  cat at pos m, mat at pos n: Q_cat * K_mat depends only on (m-n)')
print()

# For each gap, rotate cat/mat at several different absolute base positions and
# confirm the dot product stays constant -- only the gap (m-n) should matter
for gap in [1, 2, 3]:
    results = []
    for base in [0, 1, 2, 3]:
        q = rope_rotate(cat_emb, base, thetas_demo)
        k = rope_rotate(mat_emb, base + gap, thetas_demo)
        results.append(float(q @ k))
    print(f'  gap={gap}: dot products = {[f"{r:.4f}" for r in results]}  '
          f'(all equal -> {np.allclose(results, results[0], atol=1e-5)})')

print()
print('  -> RoPE guarantees relative position: the dot product depends ONLY on (m-n).')


---

## Part 4 - Queries, Keys & Values

The transformer's attention mechanism is a **soft dictionary lookup**.

| Component | Intuition | Created by |
| --------- | --------- | ---------- |
| **Q** (Query) | "What am I looking for?" | $W_Q \cdot x$ |
| **K** (Key)   | "What do I advertise?"   | $W_K \cdot x$ |
| **V** (Value) | "What do I contribute?"  | $W_V \cdot x$ |

$W_Q$, $W_K$, $W_V$ are **learned** projection matrices. Each head has its own set.


![Q, K, V data flow: query selects, key gates, value delivers](images/attention-qkv-data-flow.png)

> **PyTorch → Keras:** `torch.manual_seed(7)`, `torch.randn(D_MODEL, D_MODEL)` build random weight matrices, then `embs @ W_Q.T` projects the embeddings — a hand-rolled stand-in for a learned linear layer. **Keras/TF equivalent:** `tf.random.set_seed(7)`, `tf.random.normal((D_MODEL, D_MODEL))`, `tf.matmul(embs, W_Q, transpose_b=True)`; in real code both frameworks would use a learnable layer instead (`nn.Linear`/`tf.keras.layers.Dense`) rather than a raw random matrix.

In [ ]:
#  Q / K / V projections in 3D space
torch.manual_seed(7)
D_MODEL = 3

W_Q = torch.randn(D_MODEL, D_MODEL) * 0.5
W_K = torch.randn(D_MODEL, D_MODEL) * 0.5
W_V = torch.randn(D_MODEL, D_MODEL) * 0.5

embs = embedding_matrix[TOKEN_IDS]   # (6, 3)

# Project embeddings through each learned matrix to get the Q/K/V views
Q = embs @ W_Q.T   # (6, 3)
K = embs @ W_K.T
V = embs @ W_V.T

fig = plt.figure(figsize=(15, 4))
titles = ['Input Embeddings', 'Queries  (W_Q @ x)', 'Keys  (W_K @ x)', 'Values  (W_V @ x)']
arrays = [embs, Q, K, V]
colors = plt.cm.tab10(np.linspace(0, 0.6, SEQ_LEN))

# Plot each of the four spaces (input + Q/K/V) side by side in its own 3D panel
for idx, (title, arr) in enumerate(zip(titles, arrays)):
    ax = fig.add_subplot(1, 4, idx + 1, projection='3d')
    arr_np = arr.detach().numpy()
    for j, (word, vec) in enumerate(zip(TOKENS, arr_np)):
        ax.scatter(*vec, color=colors[j], s=70, zorder=5)
        ax.text(*vec, f' {word}', fontsize=7, color=colors[j])
    ax.set_title(title, fontsize=9, pad=6)
    ax.set_xlabel('d0', fontsize=7); ax.set_ylabel('d1', fontsize=7); ax.set_zlabel('d2', fontsize=7)
    ax.tick_params(labelsize=6)

plt.suptitle('How W_Q, W_K, W_V rotate/stretch the embedding space', fontsize=11, y=1.01)
plt.tight_layout(); plt.show()


> **PyTorch → Keras:** `Q_in @ K_in.T / math.sqrt(d_k)` computes scaled dot-product scores; `scores.masked_fill(mask.bool(), float('-inf'))` writes `-inf` into masked positions before softmax; `torch.softmax(scores, dim=-1)` normalises each row. **Keras/TF equivalent:** `tf.matmul(Q_in, K_in, transpose_b=True) / tf.sqrt(tf.cast(d_k, tf.float32))`, `tf.where(mask, tf.fill(scores.shape, float('-inf')), scores)`, and `tf.nn.softmax(scores, axis=-1)` — PyTorch's in-place `masked_fill` becomes an explicit `tf.where` since TF tensors are immutable.

In [ ]:
#  Scaled Dot-Product Attention
def scaled_attention(Q_in: torch.Tensor, K_in: torch.Tensor, V_in: torch.Tensor, mask=None):
    """Scaled dot-product attention. Returns (output, attn_weights).
    Q_in, K_in, V_in: (seq_len, d_k)
    """
    d_k = Q_in.shape[-1]

    scores = Q_in @ K_in.T / math.sqrt(d_k)
    print(f'  Raw score matrix (QK^T / sqrt(d_k)):\n  {scores.detach().numpy().round(3)}')

    if mask is not None:

        # Blocked positions get -inf so softmax assigns them zero weight
        scores = scores.masked_fill(mask.bool(), float('-inf'))

    attn_w = torch.softmax(scores, dim=-1)
    out = attn_w @ V_in
    return out, attn_w


print('=== Step-by-step attention on our sentence ===')
print(f'Q shape: {tuple(Q.shape)}  K shape: {tuple(K.shape)}  V shape: {tuple(V.shape)}\n')

out, attn_w = scaled_attention(Q, K, V)

print(f'\n  Attention weight matrix (row = query token, col = key token):')
print(f'  Tokens: {TOKENS}')
print(f'  {attn_w.detach().numpy().round(3)}')
print(f'\n  Output shape: {tuple(out.shape)}')


> **PyTorch → Keras:** `torch.triu(torch.ones(S6, S6, dtype=torch.bool), diagonal=1)` builds the upper-triangular boolean causal mask fed into `scaled_attention`. **Keras/TF equivalent:** `tf.linalg.band_part(tf.ones((S6, S6)), 0, -1) - tf.linalg.band_part(tf.ones((S6, S6)), 0, 0)` cast to bool, or the simpler `1 - tf.linalg.band_part(tf.ones((S6, S6)), -1, 0)` for the lower-triangular allowed region — TF has no single `triu` builtin outside `tf.experimental.numpy.triu`.

In [ ]:
#  Attention heatmap visualisation
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left panel: full bidirectional attention weights (no mask)
ax = axes[0]
w = attn_w.detach().numpy()
sns.heatmap(w, ax=ax, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=TOKENS, yticklabels=TOKENS, linewidths=0.5,
            cbar_kws={'label': 'attention weight'})
ax.set_title('Bidirectional Attention (encoder-style)')
ax.set_xlabel('Key token'); ax.set_ylabel('Query token'); ax.tick_params(axis='x', rotation=30)

# Right panel: same Q/K/V, but with a causal mask blocking future tokens
ax2 = axes[1]
S6 = SEQ_LEN
causal_mask_vis = torch.triu(torch.ones(S6, S6, dtype=torch.bool), diagonal=1)
_, attn_w_causal = scaled_attention(Q, K, V, mask=causal_mask_vis)
wc = attn_w_causal.detach().numpy()
sns.heatmap(wc, ax=ax2, annot=True, fmt='.2f', cmap='Oranges',
            xticklabels=TOKENS, yticklabels=TOKENS, linewidths=0.5,
            cbar_kws={'label': 'attention weight'})
ax2.set_title('Causal Attention (decoder-style)\nupper triangle masked to -inf')
ax2.set_xlabel('Key token'); ax2.set_ylabel('Query token'); ax2.tick_params(axis='x', rotation=30)

plt.suptitle('Bidirectional vs Causal Attention', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()


### Your turn - attention

You've watched attention; now drive it. Change `my_query` and **predict the top attention target before you run it**.


> **PyTorch → Keras:** `Q @ K.T / math.sqrt(...)`, `torch.softmax(scores_ex, dim=-1)`, then `.detach().numpy()` to pull the result out of the autograd graph for printing. **Keras/TF equivalent:** `tf.matmul(Q, K, transpose_b=True) / tf.sqrt(...)`, `tf.nn.softmax(scores_ex, axis=-1).numpy()` — TF eager tensors expose `.numpy()` directly with no `.detach()` step needed, since TF only tracks gradients inside an explicit `tf.GradientTape` block.

In [ ]:
#  EXERCISE 1 - attention by hand
# Change `my_query` to any token in TOKENS and PREDICT its top attention target BEFORE running.
# TOKENS = ['the', 'cat', 'sat', 'on', 'the', 'mat']
my_query = 'cat'   # try 'sat', 'mat', 'on', ...

qi = TOKENS.index(my_query)

# Recompute attention weights for just the chosen query token
scores_ex = Q @ K.T / math.sqrt(Q.shape[-1])
w_ex = torch.softmax(scores_ex, dim=-1).detach().numpy()[qi]

print(f'"{my_query}" (position {qi}) attends most to:')
for r in w_ex.argsort()[::-1][:3]:
    print(f'   {TOKENS[r]:<8} (pos {r})  weight={w_ex[r]:.3f}')


> **PyTorch → Keras:** `Q_in.detach().numpy()` bridges tensors already computed in the framework back into plain numpy so the RoPE rotation can be applied with hand-written numpy math, then `torch.tensor(rope(Q_np))` wraps the rotated arrays back into tensors. **Keras/TF equivalent:** `Q_in.numpy()` (no `.detach()` needed outside a `GradientTape`) and `tf.constant(rope(Q_np))` to convert back — the round-trip-through-numpy pattern is identical in spirit across both frameworks.

In [ ]:
#  RoPE applied to Q and K inside attention - step 1: rotate Q and K
def apply_rope_to_qk(Q_in, K_in, thetas_arr: np.ndarray) -> tuple:
    """Apply RoPE to Q and K (seq_len x d). Returns (Q_rot, K_rot) as torch.Tensor."""
    Q_np = Q_in.detach().numpy() if isinstance(Q_in, torch.Tensor) else np.asarray(Q_in)
    K_np = K_in.detach().numpy() if isinstance(K_in, torch.Tensor) else np.asarray(K_in)
    seq_len, d = Q_np.shape
    n_pairs = d // 2
    positions = np.arange(seq_len, dtype=np.float32)

    # Rotate every dimension pair of every position vector at once (vectorised RoPE)
    def rope(x):
        out_r = x.copy()
        for i in range(n_pairs):
            ang = positions * float(thetas_arr[i])
            cos_a, sin_a = np.cos(ang), np.sin(ang)
            a, b = x[:, 2*i], x[:, 2*i+1]
            out_r[:, 2*i]   = a*cos_a - b*sin_a
            out_r[:, 2*i+1] = a*sin_a + b*cos_a
        return out_r

    return torch.tensor(rope(Q_np)), torch.tensor(rope(K_np))


th_vis = np.array([1.0 / (10000 ** (2*i/D_MODEL)) for i in range(D_MODEL // 2)])
Q_raw, K_raw = Q, K
Q_rot, K_rot = apply_rope_to_qk(Q_raw, K_raw, th_vis)

print(f'{"Token":<8}  {"Q_raw":>26}  {"Q_rotated (RoPE)":>26}  {"delta norm":>10}')
print('  ' + '-' * 74)
for i, tok in enumerate(TOKENS):
    qr, qn = Q_raw[i].numpy(), Q_rot[i].numpy()
    delta = np.linalg.norm(qn - qr)
    print(f'  {tok:<8}  [{qr[0]:+.3f}, {qr[1]:+.3f}, {qr[2]:+.3f}]  '
          f'[{qn[0]:+.3f}, {qn[1]:+.3f}, {qn[2]:+.3f}]  {delta:>10.4f}')
print()
print("Notice: 'the' at position 0 has m=0, so angle=0 -> Q_rotated = Q_raw.")


#### Does that rotation actually change what attends to what?

Same projections, one difference - RoPE applied or not - side by side.


> **PyTorch → Keras:** `Q_raw @ K_raw.T / math.sqrt(D_MODEL)` followed by `torch.softmax(..., dim=-1)` recomputes attention weights with and without the RoPE-rotated Q/K for side-by-side comparison. **Keras/TF equivalent:** `tf.matmul(Q_raw, K_raw, transpose_b=True) / tf.sqrt(float(D_MODEL))` then `tf.nn.softmax(..., axis=-1)` — same scaled-dot-product-then-softmax pattern as every other attention cell in this notebook.

In [ ]:
#  RoPE in attention - step 2: the payoff on attention weights
# Compute attention scores with and without RoPE-rotated Q/K, same weights otherwise
scores_raw = Q_raw @ K_raw.T / math.sqrt(D_MODEL)
scores_rot = Q_rot @ K_rot.T / math.sqrt(D_MODEL)
attn_raw = torch.softmax(scores_raw, dim=-1)
attn_rot = torch.softmax(scores_rot, dim=-1)

# Side-by-side heatmaps isolate the effect of RoPE alone
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, w, title in zip(axes, [attn_raw, attn_rot], ['Attention WITHOUT RoPE', 'Attention WITH RoPE']):
    sns.heatmap(w.detach().numpy(), ax=ax, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=TOKENS, yticklabels=TOKENS, linewidths=0.5)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Key'); ax.set_ylabel('Query'); ax.tick_params(axis='x', rotation=30)
plt.suptitle('RoPE shifts attention weights by baking position into Q*K scores', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()


### 4c. Discovering the attention formula - softmax and sqrt(d)

#### Predict first

1. Raw $QK^T$ scores can be negative and don't sum to 1. **What single operation turns an arbitrary score vector into a probability distribution?**
2. In a real model $d_k$ is 64-128. The dot product of two random vectors grows with dimension. **What happens to softmax when scores get very large?**


> **PyTorch → Keras:** `(Q_rot @ K_rot.T)[0].detach().numpy()` computes the raw (pre-softmax) score row for one query and pulls it out of the autograd graph to print. **Keras/TF equivalent:** `tf.matmul(Q_rot, K_rot, transpose_b=True)[0].numpy()` — TF eager tensors are numpy-convertible without an explicit detach step.

In [ ]:
#  DISCOVERING softmax and the sqrt(d) scale - decision 1: why softmax?
# Pull out the raw (pre-softmax) score row for one query token
raw = (Q_rot @ K_rot.T)[0].detach().numpy()
print('Raw QK^T scores for one query row:')
print('  ', raw.round(3))
print(f'  sum = {raw.sum():+.3f}  (not 1)  and some are negative -> NOT a probability.')
print('  softmax fixes both: exp() makes them positive, then normalise to sum = 1.')
print()
print('  -> Softmax is the only differentiable function that produces a probability')
print('     distribution from arbitrary real-valued scores. -> conclusion')


#### Decision 2 - why divide by sqrt(d)?

Softmax alone isn't enough. The dot product of two random vectors has variance that **grows with dimension**.


> **PyTorch → Keras:** `torch.randn(4000, d)`, `.sum(dim=-1)`, `.std()`, and `torch.softmax(s_un, dim=-1).max(dim=-1).values.mean()` measure how dot-product variance and softmax peak-probability scale with dimension. **Keras/TF equivalent:** `tf.random.normal((4000, d))`, `tf.reduce_sum(..., axis=-1)`, `tf.math.reduce_std(...)`, and `tf.reduce_max(tf.nn.softmax(s_un, axis=-1), axis=-1)` — PyTorch's `dim=` keyword is TF's `axis=` throughout.

In [ ]:
#  Decision 2, step 1: dot-product variance grows with dimension
dims = [4, 16, 64, 256, 1024]
print(f'{"d_k":>6} | {"std(QK^T) unscaled":>18} | {"std after /sqrt(d_k)":>15}')
print('  ' + '-' * 46)
peak_unscaled, peak_scaled = [], []

# For each dimension, measure raw dot-product spread and how sharply softmax
# peaks, both scaled and unscaled by sqrt(d)
for d in dims:
    q_v = torch.randn(4000, d)
    k_v = torch.randn(4000, d)
    dot = (q_v * k_v).sum(dim=-1).numpy()
    print(f'{d:>6} | {dot.std():>18.2f} | {(dot / math.sqrt(d)).std():>15.2f}')
    qr2 = torch.randn(500, 8, d)
    kr2 = torch.randn(500, 8, d)
    s_un = (qr2 * kr2).sum(dim=-1)
    s_sc = s_un / math.sqrt(d)
    peak_unscaled.append(float(torch.softmax(s_un, dim=-1).max(dim=-1).values.mean()))
    peak_scaled.append(float(torch.softmax(s_sc, dim=-1).max(dim=-1).values.mean()))
print('  Unscaled variance grows like sqrt(d); dividing by sqrt(d) pins it ~1 at every width.')


#### The consequence - saturation kills the gradient

Big scores push softmax toward a one-hot spike. A one-hot softmax has almost no slope, so the gradient vanishes.


> **PyTorch → Keras:** `torch.randn(8, d_big, requires_grad=True)` creates a leaf tensor that tracks gradients; `loss_sat.backward()` runs autograd back through the softmax; `q_sat.grad` reads the accumulated gradient afterward. **Keras/TF equivalent:** there's no `requires_grad` flag — instead you open a `with tf.GradientTape() as tape:` block, compute the loss inside it, then call `tape.gradient(loss_sat, q_sat)` to get the gradient explicitly (TF never mutates a `.grad` attribute in place).

In [ ]:
#  Decision 2, step 2: the CONSEQUENCE - saturation kills the gradient
d_big = 256
k_sat = torch.randn(8, d_big)

# Build a fresh leaf tensor each time so gradients aren't accumulated across the two conditions
for label, scale in [('WITHOUT /sqrt(d)', 1.0), ('WITH /sqrt(d)', math.sqrt(d_big))]:
    q_sat = torch.randn(8, d_big, requires_grad=True)
    p = torch.softmax((q_sat @ k_sat.T) / scale, dim=-1)
    loss_sat = torch.sum(torch.max(p, dim=-1).values)
    loss_sat.backward()
    g = q_sat.grad
    peak_p = torch.max(p, dim=-1).values.mean().item()
    print(f'  {label:<16}: mean peak prob = {peak_p:.3f}   |grad(q)| = {torch.norm(g).item():.2e}')

print()
print('WITHOUT scaling: softmax -> one-hot (peak~1) -> gradient ~0 -> layer cannot learn.')
print('WITH   scaling: distribution stays soft -> gradient flows -> training works.')

# Plot how softmax's peak probability grows with dimension, with vs. without scaling
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(dims, peak_unscaled, 'o-', color='tomato',   lw=2, label='unscaled QK^T')
ax.plot(dims, peak_scaled,   'o-', color='seagreen', lw=2, label='scaled QK^T/sqrt(d)')
ax.axhline(1/8, color='gray', ls='--', lw=1, label='uniform (1/8)')
ax.set_xscale('log', base=2); ax.set_xlabel('d_k  (attention head dimension)')
ax.set_ylabel('mean peak softmax probability')
ax.set_title('Without sqrt(d) scaling, softmax saturates to one-hot as d_k grows')
ax.set_ylim(0, 1.05); ax.legend()
plt.tight_layout(); plt.show()


---

## Part 5 - Multi-Head Attention

**Multi-Head Attention** (MHA) runs $H$ parallel attention heads:

$$\text{MHA}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_H) \cdot W_O$$

**Why multiple heads?** Each head can specialise on a different relationship type.

Each head receives the token's **full** $d_{model}$-dimensional representation. Its own learned $W_Q^{(h)}$, $W_K^{(h)}$, and $W_V^{(h)}$ projections map that representation into a lower-dimensional $d_{head}$ subspace, where it computes one attention pattern. Training may lead different heads to emphasize syntax, position, entity relations, or other features, but those roles are **learned rather than assigned**, can overlap, and are not guaranteed to remain cleanly human-interpretable.

> **PyTorch → Keras:** `class MultiHeadAttention(nn.Module)` subclasses `nn.Module`, defines its learnable sub-layers (`nn.Linear(d_model, d_model, bias=False)`) in `__init__`, and implements the computation in `forward()`; `.reshape(...).transpose(1, 2)` splits the model dimension into heads and swaps the head/sequence axes. **Keras/TF equivalent:** subclass `tf.keras.layers.Layer`, create `tf.keras.layers.Dense(d_model, use_bias=False)` sub-layers in `__init__`, and implement the computation in `call()` instead of `forward()`; `tf.transpose(tf.reshape(x, (B, S, H, d_head)), perm=[0, 2, 1, 3])` is the equivalent reshape-then-swap-axes step.

In [ ]:
#  Working model constants
D_WORK = 16    # functional model dimension
NUM_HEADS = 2  # attention heads
D_HEAD = D_WORK // NUM_HEADS   # 8 per head
D_FF = 32      # feed-forward hidden size


# Splits Q/K/V into n_heads parallel attention heads, then concatenates and projects back
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.d_model = d_model
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    # Reshape into (B, H, S, d_head), run scaled dot-product attention per head,
    # then merge the heads back together
    def forward(self, x, mask=None):
        B, S, _ = x.shape
        Q_mh = self.W_Q(x).reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)  # (B, H, S, d_head)
        K_mh = self.W_K(x).reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)
        V_mh = self.W_V(x).reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)
        scores = (Q_mh @ K_mh.transpose(-2, -1)) / math.sqrt(self.d_head)  # (B, H, S, S)
        if mask is not None:
            scores = scores.masked_fill(mask.bool(), float('-inf'))
        attn_w_mh = torch.softmax(scores, dim=-1)   # (B, H, S, S)
        out = attn_w_mh @ V_mh                       # (B, H, S, d_head)
        out = out.transpose(1, 2).reshape(B, S, self.d_model)
        return self.W_O(out), attn_w_mh


#  Demo
torch.manual_seed(42)
mha = MultiHeadAttention(D_WORK, NUM_HEADS)

proj = nn.Linear(D_MODEL, D_WORK, bias=False)

# Project the 3D toy embeddings up to the working 16-dim model space (no gradient needed for this demo)
with torch.no_grad():
    x_work = proj(embs).unsqueeze(0)   # (1, 6, 16)

mha_out, head_weights = mha(x_work)
print(f'MHA output shape: {tuple(mha_out.shape)}   head_weights shape: {tuple(head_weights.shape)}')

# Plot each head's attention pattern in its own panel
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for h in range(NUM_HEADS):
    ax = axes[h]
    w_h = head_weights[0, h].detach().numpy()
    sns.heatmap(w_h, ax=ax, annot=True, fmt='.2f', cmap='Purples',
                xticklabels=TOKENS, yticklabels=TOKENS, linewidths=0.5, cbar=False)
    ax.set_title(f'Head {h} attention weights')
    ax.set_xlabel('Key'); ax.set_ylabel('Query'); ax.tick_params(axis='x', rotation=30)
plt.suptitle('Multi-Head Attention - each head learns a different relationship', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()


#### Predict first - can one head do two jobs?

A single attention head produces **one** score matrix. Suppose you want every token to attend to **both** its previous token and its most semantically similar token.

**Predict:** can one head satisfy both relations at once, or must it pick one?


> **PyTorch → Keras:** `torch.manual_seed(0)`, `X @ torch.randn(3, 3)` projects the embeddings through a random matrix, `torch.tensor(scores_pos)` wraps a hand-built numpy score matrix, and `torch.softmax(..., dim=-1)` turns each into an attention distribution. **Keras/TF equivalent:** `tf.random.set_seed(0)`, `tf.matmul(X, tf.random.normal((3, 3)))`, `tf.constant(scores_pos)`, and `tf.nn.softmax(..., axis=-1)` — the same "hand-craft two attention patterns to prove they're independent" technique translates operation-for-operation.

In [ ]:
#  PROVING the multi-head claim - step 1: two heads, two patterns
torch.manual_seed(0)
S = SEQ_LEN
X = embs   # (S, 3)
V_shared = X @ torch.randn(3, 3)

# Relation P (positional): attend to the PREVIOUS token
prev_idx = np.array([max(i-1, 0) for i in range(S)])
scores_pos = np.full((S, S), -9.0, dtype=np.float32)
for i in range(S):
    scores_pos[i, prev_idx[i]] = 9.0
A_pos = torch.softmax(torch.tensor(scores_pos), dim=-1)

# Relation C (content): attend to the most SEMANTICALLY SIMILAR token
sim = (X @ X.T).detach().numpy()
np.fill_diagonal(sim, -1e9)
near_idx = sim.argmax(-1)
scores_con = np.full((S, S), -9.0, dtype=np.float32)
for i in range(S):
    scores_con[i, near_idx[i]] = 9.0
A_con = torch.softmax(torch.tensor(scores_con), dim=-1)

corr = np.corrcoef(A_pos.numpy().flatten(), A_con.numpy().flatten())[0, 1]
print(f'Correlation between the two head patterns: {corr:+.3f}   (~0 -> different information)')

# Plot both head patterns side by side for direct comparison
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, A, title, cmap in [
    (axes[0], A_pos.numpy(), 'Head P - positional (attend to previous token)', 'Greens'),
    (axes[1], A_con.numpy(), 'Head C - content (attend to most similar token)', 'Purples'),
]:
    sns.heatmap(A, ax=ax, annot=True, fmt='.2f', cmap=cmap,
                xticklabels=TOKENS, yticklabels=TOKENS, linewidths=0.5, cbar=False)
    ax.set_title(title, fontsize=10); ax.set_xlabel('Key'); ax.set_ylabel('Query')
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('Two heads, two DIFFERENT relations', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()


#### So they differ - but could a *single* head carry both?

Each head yields exactly **one** output per token. Let's measure how well each head recovers each relation.


> **PyTorch → Keras:** `A_pos @ V_shared` and `A_con @ V_shared` are plain tensor matmuls that blend values by attention weight, reused directly from the tensors built in the previous cell. **Keras/TF equivalent:** `tf.matmul(A_pos, V_shared)` / `tf.matmul(A_con, V_shared)` — no framework-specific nuance here beyond PyTorch's `@` operator mapping straight onto `tf.matmul`.

In [ ]:
#  PROVING the multi-head claim - step 2: one head can't do both
# Reconstruct each relation's target values and this head's actual blended output
target_prev = V_shared[prev_idx]
target_near = V_shared[near_idx]
out_pos = A_pos @ V_shared
out_con = A_con @ V_shared


# Mean squared error between two tensors
def mse(a, b):
    return float(((a - b) ** 2).mean())


print('Reconstruction error (lower = that relation is captured):')
print(f'  Head P alone -> previous-token target : {mse(out_pos, target_prev):.4f}   <- nails it')
print(f'  Head P alone -> similar-token  target : {mse(out_pos, target_near):.4f}   <- misses it')
print(f'  Head C alone -> previous-token target : {mse(out_con, target_prev):.4f}   <- misses it')
print(f'  Head C alone -> similar-token  target : {mse(out_con, target_near):.4f}   <- nails it')
print()
print('  -> A single head serves ONE relation well, never both.')
print('  -> Concatenating [Head P ; Head C] delivers BOTH targets in parallel.')
print('  -> That is why H heads exist: H independent relations, computed at once.')


### Your turn - heads

Dial the number of heads up and down and watch how independent their patterns become.


> **PyTorch → Keras:** `torch.manual_seed(42)` reseeds before instantiating a fresh `MultiHeadAttention(D_WORK, n_heads)` module and calling it on `x_work`; `.reshape(n_heads, -1)` flattens each head's pattern for correlation analysis. **Keras/TF equivalent:** `tf.random.set_seed(42)`, a fresh `MultiHeadAttention` `tf.keras.layers.Layer` instance called on `x_work`, and `tf.reshape(w_ex[0], (n_heads, -1))` — identical module-reuse pattern, different base class.

In [ ]:
#  EXERCISE 2 - how many heads?
# Change `n_heads` (must divide D_WORK = 16: try 1, 2, 4, 8).
# Predict: more heads = more independent relations captured at once.
n_heads = 1   # try 1, then 4, then 8

torch.manual_seed(42)
mha_ex = MultiHeadAttention(D_WORK, n_heads)
_, w_ex = mha_ex(x_work)
print(f'{n_heads} head(s) -> {n_heads} attention pattern(s), each {D_WORK // n_heads}-dim wide.')

# Flatten each head's attention pattern for pairwise correlation
pats = w_ex[0].detach().reshape(n_heads, -1).numpy()

# Only compare pairs when there's more than one head to compare
if n_heads > 1:
    corrs = np.corrcoef(pats)
    print('Pairwise correlations between head patterns:')
    for hi in range(n_heads):
        for hj in range(hi + 1, n_heads):
            print(f'  head {hi} vs head {hj}: r = {corrs[hi, hj]:+.3f}')
else:
    print('  (only one head - no pairwise comparison)')


---

## Part 6 - Feed-Forward Network & Layer Normalisation

### Feed-Forward Network (FFN)

$$\text{FFN}(x) = \text{GELU}(x W_1 + b_1)\, W_2 + b_2$$

Expands by 4x then projects back. Adds non-linear transformation capacity.

### Layer Normalisation

Applied **before** each sub-layer (Pre-LN style). Normalises each token's vector to zero mean and unit variance.


> **PyTorch → Keras:** `nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))` chains layers declaratively, and `nn.LayerNorm(D_WORK, eps=1e-5)` normalises each token's activation vector. **Keras/TF equivalent:** `tf.keras.Sequential([tf.keras.layers.Dense(d_ff), tf.keras.layers.Activation('gelu'), tf.keras.layers.Dense(d_model)])` and `tf.keras.layers.LayerNormalization(epsilon=1e-5)` — both frameworks provide the same "stack of layers" container and a built-in LayerNorm with matching default epsilon semantics.

In [ ]:
#  FeedForward + LayerNorm
# Standard transformer feed-forward block: expand 4x, GELU, project back
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)


#  Visualise LayerNorm effect
torch.manual_seed(42)
ffn = FeedForward(D_WORK, D_FF)
norm = nn.LayerNorm(D_WORK, eps=1e-5)

x_raw = x_work[0]   # (6, 16)

# Run the FFN then LayerNorm without tracking gradients (visualisation only)
with torch.no_grad():
    x_after = ffn(x_raw)       # (6, 16) raw FFN output
    x_normed = norm(x_after)   # (6, 16) after LayerNorm

# Plot each stage's per-token activation profile side by side
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, data, title in zip(axes, [x_raw, x_after, x_normed],
                           ['Input to FFN', 'FFN output (raw)', 'After LayerNorm']):
    data_np = data.detach().numpy()
    for j, token in enumerate(TOKENS):
        vals = data_np[j]
        ax.plot(vals, alpha=0.7, label=f'{token}  mu={vals.mean():.2f}, s={vals.std():.2f}')
    ax.set_title(title); ax.set_xlabel('Hidden dimension'); ax.set_ylabel('Activation value')
    ax.legend(fontsize=7); ax.axhline(0, color='black', lw=0.5, ls='--')
plt.suptitle('FFN activations before and after LayerNorm', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

print('LayerNorm centres and normalises each token slice.')
print('Mean and std across dimensions after LN:')
x_normed_np = x_normed.detach().numpy()
for j, token in enumerate(TOKENS):
    v = x_normed_np[j]
    print(f'  {token:<8}  mean={v.mean():+.4f}  std={v.std():.4f}')


![Complete Transformer block: LayerNorm → Multi-Head Attention → Residual → LayerNorm → FFN → Residual](images/transformer-block-overview.png)

---

## Part 7 - Full Transformer Block & RNN Comparison

A single **Transformer Block** wires MHA + FFN together with layer norm and residuals:

```
x --> LayerNorm --> MHA --> (+x) --> LayerNorm --> FFN --> (+x) --> output
```

Stack $L$ of these blocks = the full encoder/decoder stack.


> **PyTorch → Keras:** `class TransformerBlock(nn.Module)` composes previously-defined `nn.Module` sub-layers (`nn.LayerNorm`, `MultiHeadAttention`, `FeedForward`) inside `__init__` and wires them with residual adds (`x = x + mha_out`) in `forward()`. **Keras/TF equivalent:** subclass `tf.keras.layers.Layer`, create the same sub-layers in `__init__`, and wire them in `call()` — the residual-add line `x + mha_out` is identical Python syntax since TF tensors overload `+` the same way.

In [ ]:
#  TransformerBlock
# Pre-LN transformer block: LayerNorm -> MHA -> residual, LayerNorm -> FFN -> residual
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model, eps=1e-5)
        self.mha   = MultiHeadAttention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-5)
        self.ffn   = FeedForward(d_model, d_ff)

    def forward(self, x, mask=None):

        # Pre-LN attention sublayer, then pre-LN feed-forward sublayer, both with residual adds
        mha_out, attn_w_b = self.mha(self.norm1(x), mask=mask)
        x = x + mha_out
        x = x + self.ffn(self.norm2(x))
        return x, attn_w_b


torch.manual_seed(42)
block1 = TransformerBlock(D_WORK, NUM_HEADS, D_FF)
block2 = TransformerBlock(D_WORK, NUM_HEADS, D_FF)

x0 = x_work.clone()   # (1, 6, 16)

# Run two stacked blocks without tracking gradients (inspection only)
with torch.no_grad():
    x1, aw1 = block1(x0)
    x2, aw2 = block2(x1)

print('Input -> Block 1 -> Block 2:')
print(f'  x0: {tuple(x0.shape)}  norm={float(torch.norm(x0)):.3f}')
print(f'  x1: {tuple(x1.shape)}  norm={float(torch.norm(x1)):.3f}')
print(f'  x2: {tuple(x2.shape)}  norm={float(torch.norm(x2)):.3f}')

# Per-token representation norm at each stage of the stack
fig, ax = plt.subplots(figsize=(8, 4))
norms = {
    'Layer 0 (input)': torch.norm(x0[0], dim=-1).detach().numpy(),
    'Layer 1 output':  torch.norm(x1[0], dim=-1).detach().numpy(),
    'Layer 2 output':  torch.norm(x2[0], dim=-1).detach().numpy(),
}
x_pos = np.arange(SEQ_LEN); width = 0.25

# Plot grouped bars comparing token norms across layers
for k, (label, vals) in enumerate(norms.items()):
    ax.bar(x_pos + k*width, vals, width, label=label, alpha=0.85)
ax.set_xticks(x_pos + width); ax.set_xticklabels(TOKENS)
ax.set_ylabel('Representation L2 norm')
ax.set_title('Token representations grow through transformer blocks')
ax.legend(); plt.tight_layout(); plt.show()

#### Predict first - does the skip connection really matter?

We'll stack **24** simple layers and read the gradient that reaches **layer 1** (furthest from the loss), with and without the skip $x + \text{SubLayer}(x)$.

**Predict:** without the skip, will the gradient at layer 1 be vanishingly small?


> **PyTorch → Keras:** `nn.ModuleList([ProbeBlock(d, residual) for _ in range(DEPTH)])` registers a list of sub-modules so their parameters are tracked; `loss_p.backward()` then `b.lin.weight.grad.norm()` reads the per-layer gradient magnitude after backprop. **Keras/TF equivalent:** a plain Python list of `tf.keras.layers.Layer` instances (or `tf.keras.Sequential`) works the same way for tracking; gradients come from `with tf.GradientTape() as tape: ...; grads = tape.gradient(loss_p, [b.lin.kernel for b in blocks])` instead of an in-place `.grad` attribute.

In [ ]:
#  DEMONSTRATING why residual connections make depth trainable

DEPTH = 24
d = D_WORK


# Minimal block for probing gradient flow with vs. without a residual skip
class ProbeBlock(nn.Module):
    def __init__(self, d_in, residual):
        super().__init__()
        self.lin = nn.Linear(d_in, d_in)
        self.residual = residual

    def forward(self, x):
        y = torch.tanh(self.lin(x))

        # Toggle point: with residual, add the input back; without, it's a plain deep MLP
        return x + y if self.residual else y


# Run a DEPTH-layer stack and return the gradient norm reaching each layer
def gradient_reaching_each_layer(residual):
    torch.manual_seed(0)
    blocks = nn.ModuleList([ProbeBlock(d, residual) for _ in range(DEPTH)])
    x_p = torch.randn(1, d)
    h = x_p
    for b in blocks:
        h = b(h)
    loss_p = torch.mean(h ** 2)
    loss_p.backward()

    # Gradient norm on each block's weight matrix, ordered from earliest to latest layer
    return [b.lin.weight.grad.norm().item() for b in blocks]


g_res   = gradient_reaching_each_layer(residual=True)
g_plain = gradient_reaching_each_layer(residual=False)

fig, ax = plt.subplots(figsize=(9, 4.2))

# Plot gradient norm per layer on a log scale, with vs. without residual connections
ax.plot(range(1, DEPTH+1), g_plain, 'o-', color='tomato',   lw=2, label='WITHOUT residual')
ax.plot(range(1, DEPTH+1), g_res,   'o-', color='seagreen', lw=2, label='WITH residual')
ax.set_yscale('log')
ax.set_xlabel('Layer (1 = furthest from loss, closest to input)')
ax.set_ylabel('|gradient| reaching this layer  (log scale)')
ax.set_title('Residual connections keep gradients alive all the way to layer 1')
ax.legend(); plt.tight_layout(); plt.show()

print('Gradient norm reaching layer 1 (the earliest, hardest-to-train layer):')
print(f'  WITHOUT residual: {g_plain[0]:.2e}   <- vanished')
print(f'  WITH    residual: {g_res[0]:.2e}   <- healthy')
ratio = g_res[0] / max(g_plain[0], 1e-30)
print(f'  The skip path delivers ~{ratio:.1e}x more gradient to the earliest layer.')

### Your turn - depth

Push the stack deeper and watch the no-residual gradient collapse while the residual one stays alive.


In [ ]:
#  EXERCISE 3 - how deep can you go WITHOUT residuals?
# Change `DEPTH` (try 8, 24, 60) and PREDICT how far the gradient survives.
DEPTH = 40

g_res_ex   = gradient_reaching_each_layer(residual=True)
g_plain_ex = gradient_reaching_each_layer(residual=False)
print(f'At depth {DEPTH}, gradient reaching layer 1 (the hardest to train):')
print(f'  WITHOUT residual: {g_plain_ex[0]:.2e}')
print(f'  WITH    residual: {g_res_ex[0]:.2e}')
ratio_ex = g_res_ex[0] / max(g_plain_ex[0], 1e-30)
print(f'  -> Ratio: {ratio_ex:.1e}x more gradient with residuals.')


---

## Part 1 Checkpoint: The Reusable Block Is Ready

You now have the complete reusable mechanism:

```text
token IDs -> embeddings + position -> multi-head attention -> residual -> FFN -> residual
```

The remaining architectural choices are about **visibility and training purpose**, not a different attention primitive.

- [Continue to Part 2](02-decoder-only-language-model.ipynb): add a causal mask, next-token loss, and autoregressive generation.
- [Jump to Part 3](03-encoder-decoder-and-cross-attention.ipynb): remove the source-side causal mask and let a decoder query the encoded source through cross-attention.
